# 01b. Financial Analysis: Fundamentals & Ratios

## Objective
To analyze the financial health of Tata Motors using Balance Sheet, Income Statement, and Cash Flow data.

## Scope
- **Financials:** 5-Year Trend Analysis.
- **Ratios:** P/E, P/B, Debt/Equity, ROE.
- **Valuation:** Is the stock overvalued vs its fundamentals?

> **Important Note:** We acknowledge the recent corporate actions (Dividend Splits/DVR cancellation). While `yfinance` attempts to adjust historical data, per-share metrics should be interpreted with this context.

In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Fetch Financial Data (Robust)

In [2]:
def fetch_financials(ticker):
    print(f'Fetching financials for {ticker}...')
    stock = yf.Ticker(ticker)
    
    # Try fetching
    bs = stock.balance_sheet
    if bs.empty and '.NS' in ticker:
        print('NSE data empty, trying BSE...')
        stock = yf.Ticker(ticker.replace('.NS', '.BO'))
        bs = stock.balance_sheet
        
    return stock

ticker = 'TATAMOTORS.NS'
stock = fetch_financials(ticker)

# Fetch Annual Financials
balance_sheet = stock.balance_sheet
income_stmt = stock.income_stmt
cashflow = stock.cashflow

# Transpose for easier plotting (Years as Index)
balance_sheet = balance_sheet.T
income_stmt = income_stmt.T
cashflow = cashflow.T

# Display latest
if not income_stmt.empty:
    print(income_stmt.head())
else:
    print('Warning: Financial data could not be fetched.')

Fetching financials for TATAMOTORS.NS...
NSE data empty, trying BSE...


## 3. Profitability Analysis

Tracking Revenue vs Net Income to see if growth is translating to profit.

In [3]:
if not income_stmt.empty:
    plt.figure(figsize=(12, 6))
    try:
        # Loose matching for columns
        cols = income_stmt.columns.astype(str)
        rev_col = [c for c in cols if 'Total Revenue' in c or 'Revenue' in c]
        net_inc_col = [c for c in cols if 'Net Income' in c or 'Net Profit' in c]
        
        if rev_col and net_inc_col:
            income_stmt[rev_col[0]].plot(kind='bar', color='blue', alpha=0.6, label='Revenue')
            income_stmt[net_inc_col[0]].plot(kind='line', color='green', marker='o', secondary_y=True, label='Net Income')
            plt.title('Revenue vs Net Income Trends')
            plt.legend()
            plt.show()
        else:
            print('Required columns for profitability plot not found.')
    except Exception as e:
        print(f'Plotting Error: {e}')

## 4. Solvency (Debt Interpretation)

In [4]:
if not balance_sheet.empty:
    # Debt to Equity
    try:
        # Helper to find column safely
        def get_col(df, keywords):
            matches = [c for c in df.columns.astype(str) if any(k in c for k in keywords)]
            return df[matches[0]] if matches else pd.Series(0, index=df.index)
            
        total_debt = get_col(balance_sheet, ['Total Debt', 'Long Term Debt'])
        total_equity = get_col(balance_sheet, ['Stockholders Equity', 'Total Equity'])
        
        if not total_equity.eq(0).all():
            d_e_ratio = total_debt / total_equity
            
            plt.figure(figsize=(10, 5))
            d_e_ratio.plot(kind='bar', color='red')
            plt.title('Debt-to-Equity Ratio Trend')
            plt.ylabel('Ratio')
            plt.axhline(1, color='black', linestyle='--', label='Safe Threshold (1.0)')
            plt.legend()
            plt.show()
        else:
            print('Equity data missing or zero, cannot calculate D/E ratio.')
            
    except Exception as e:
        print(f'Debt Analysis Error: {e}')

## 5. Export Fundamentals

In [5]:
DATA_DIR = '../data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

balance_sheet.to_csv(os.path.join(DATA_DIR, 'tata_motors_balance_sheet.csv'))
income_stmt.to_csv(os.path.join(DATA_DIR, 'tata_motors_income_stmt.csv'))
cashflow.to_csv(os.path.join(DATA_DIR, 'tata_motors_cashflow.csv'))
print('Saved financial data.')

Saved financial data.
